# Notebook 4: Signal Generation + Backtest

Applies the train-set-fitted parameters (OU process, entry threshold) to generate
trading signals, then reports performance **separately for train (in-sample) and
test (out-of-sample)** periods. Benchmarks the HYSYS partial spread strategy against
the generic 3:2:1 crack spread to demonstrate the HYSYS-derived weighting actually
improves performance, not just that it is physically motivated.

## Look-ahead bias handling

All PnL is computed with the signal **lagged by one trading day**
(`signal.shift(1)`): a position entered on day *t* is based on the z-score computed
from the closing price on day *t-1*, and the trade is assumed executed at day *t*'s
close. In other words, **trades are entered the day after the close used to compute
the signal** - the strategy never trades on information not yet available at the
time of the decision.

## Operational inertia friction term

A refinery cannot instantly change its yield slate - feed heater temperatures,
column draw stages, and side stripper steam rates require days of operational
adjustment. A **minimum 5-day holding period** is enforced between signal changes,
derived directly from the HYSYS process simulation.

## Transaction cost assumption

**$0.05/bbl per trade.** NYMEX RBOB and Heating Oil futures contracts are sized at
42,000 gallons (1,000 bbl) per contract. Typical bid/ask spreads on the front-month
RBOB and HO contracts run roughly $0.0005-$0.001/gallon (~$0.02-$0.04/bbl-equivalent)
in normal liquidity, and WTI futures (CL) typically show a one-tick ($0.01/bbl)
spread. Combining the round-trip cost of establishing offsetting positions across
three legs (crude short/long + two product legs) plus modest slippage on execution,
**$0.05/bbl per trade is a deliberately conservative (i.e. higher-than-typical)
estimate** intended to avoid overstating strategy profitability. It does not include
exchange/clearing fees (typically a further ~$1-2 per contract, i.e. ~$0.001-0.002/bbl,
immaterial at this scale) or margin financing costs.

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from signal_generator import (
    generate_signals, apply_holding_period, compute_pnl,
    compute_performance_metrics, compute_generic_321_spread
)
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Load data with train/test split flag, and the fixed parameters from Notebook 3
df = pd.read_csv('../data/spread_full_with_split.csv', index_col=0, parse_dates=True)
params = pd.read_csv('../data/ou_parameters.csv', index_col=0)['value']

ENTRY_THRESHOLD  = float(params['entry_threshold'])   # fixed from train-set sensitivity analysis (NB3)
EXIT_THRESHOLD   = 0.0
LOOKBACK         = int(float(params['lookback']))
MIN_HOLDING_DAYS = 5
TRANSACTION_COST = 0.05
split_idx        = int(float(params['split_idx']))

print('Parameters fixed in Notebook 3 (train set), applied unchanged here:')
print(f'  Entry threshold:    +/-{ENTRY_THRESHOLD} sigma')
print(f'  Lookback window:    {LOOKBACK} days')
print(f'  Min holding period: {MIN_HOLDING_DAYS} days')
print(f'  Transaction cost:   ${TRANSACTION_COST}/bbl per trade')
print(f'  Train/test split index: {split_idx} ({df.index[split_idx].date()})')

In [ ]:
# ============================================================
# Z-SCORE - computed over the FULL series using only the fixed
# 252-day rolling window (a mechanical, backward-looking
# calculation - not re-fitted on test data). At any date t this
# uses only data up to and including t, so it does not leak
# information from the test set into the train set or vice versa.
# ============================================================

df['margin_mean'] = df['spread_hysys'].rolling(LOOKBACK).mean()
df['margin_std']  = df['spread_hysys'].rolling(LOOKBACK).std()
df['zscore']      = (df['spread_hysys'] - df['margin_mean']) / df['margin_std']
df = df.dropna(subset=['zscore'])

# Recompute split index after dropping NaN rows from the rolling window warm-up
split_date = pd.Timestamp(params['split_date'])
train_mask = df.index < split_date
test_mask  = df.index >= split_date

print(f'After dropping rolling-window warm-up NaNs:')
print(f'  Train: {train_mask.sum()} obs')
print(f'  Test:  {test_mask.sum()} obs')

In [ ]:
# ============================================================
# SIGNAL GENERATION - applied to the FULL series using the
# threshold fixed in Notebook 3. No parameter is re-fit here.
# ============================================================

df['signal_raw'] = generate_signals(df['zscore'], ENTRY_THRESHOLD, EXIT_THRESHOLD)
df['signal']     = apply_holding_period(df['signal_raw'], MIN_HOLDING_DAYS)

# Generic 3:2:1 benchmark spread + its own z-score / signal, same parameters,
# for an apples-to-apples comparison
df['margin_mean_321'] = df['crack_321'].rolling(LOOKBACK).mean()
df['margin_std_321']  = df['crack_321'].rolling(LOOKBACK).std()
df['zscore_321']      = (df['crack_321'] - df['margin_mean_321']) / df['margin_std_321']
df['signal_321_raw']  = generate_signals(df['zscore_321'], ENTRY_THRESHOLD, EXIT_THRESHOLD)
df['signal_321']      = apply_holding_period(df['signal_321_raw'], MIN_HOLDING_DAYS)

print('Signal generation complete for both HYSYS strategy and 3:2:1 benchmark.')

In [ ]:
# ============================================================
# PnL - HYSYS strategy (1-day signal lag, see methodology note above)
# ============================================================

pnl_df = compute_pnl(df['signal'], df['spread_hysys'], TRANSACTION_COST)
df['pnl_net_hysys']      = pnl_df['pnl_net']
df['equity_curve_hysys'] = pnl_df['equity_curve']
df['trade_flag_hysys']   = pnl_df['trade_flag']

# PnL - 3:2:1 benchmark (identical mechanics, same lag, same costs)
pnl_df_321 = compute_pnl(df['signal_321'], df['crack_321'], TRANSACTION_COST)
df['pnl_net_321']      = pnl_df_321['pnl_net']
df['equity_curve_321'] = pnl_df_321['equity_curve']
df['trade_flag_321']   = pnl_df_321['trade_flag']

print('PnL computed for both strategies.')

In [ ]:
# ============================================================
# PERFORMANCE METRICS - TRAIN vs TEST, HYSYS vs 3:2:1
# ============================================================

def full_metrics(pnl, equity, trade_flag, n_days):
    m = compute_performance_metrics(pnl, equity)
    num_trades = trade_flag.sum() / 2
    turnover   = trade_flag.sum() / n_days   # trades per day, simple turnover proxy
    m['num_trades'] = num_trades
    m['turnover']   = turnover
    return m

rows = []
for label, mask in [('Train (in-sample)', train_mask), ('Test (out-of-sample)', test_mask)]:
    for strat_label, pnl_col, eq_col, trade_col in [
        ('HYSYS partial spread', 'pnl_net_hysys', 'equity_curve_hysys', 'trade_flag_hysys'),
        ('Generic 3:2:1 (benchmark)', 'pnl_net_321', 'equity_curve_321', 'trade_flag_321')
    ]:
        sub_pnl   = df.loc[mask, pnl_col]
        sub_eq    = sub_pnl.cumsum()
        sub_trade = df.loc[mask, trade_col]
        m = full_metrics(sub_pnl, sub_eq, sub_trade, mask.sum())
        rows.append({
            'Period': label,
            'Strategy': strat_label,
            'Total Return ($/bbl)': round(m['total_return'], 2),
            'Sharpe Ratio': round(m['sharpe_ratio'], 2),
            'Annualised Vol ($/bbl)': round(m['annualised_vol'], 2),
            'Max Drawdown ($/bbl)': round(m['max_drawdown'], 2),
            'Win Rate': f"{m['win_rate']:.1%}",
            'Num Trades': int(m['num_trades']),
            'Turnover (trades/day)': round(m['turnover'], 4)
        })

results_table = pd.DataFrame(rows)
print('=' * 100)
print('PERFORMANCE SUMMARY - TRAIN vs TEST, HYSYS STRATEGY vs 3:2:1 BENCHMARK')
print('=' * 100)
results_table

In [ ]:
# Highlight the key comparison: out-of-sample HYSYS vs out-of-sample 3:2:1
test_results = results_table[results_table['Period'] == 'Test (out-of-sample)']
print('KEY RESULT - Out-of-sample performance comparison:')
print(test_results.to_string(index=False))

hysys_sharpe = test_results[test_results['Strategy'] == 'HYSYS partial spread']['Sharpe Ratio'].values[0]
bench_sharpe = test_results[test_results['Strategy'] == 'Generic 3:2:1 (benchmark)']['Sharpe Ratio'].values[0]

print(f"\nHYSYS strategy out-of-sample Sharpe: {hysys_sharpe}")
print(f"3:2:1 benchmark out-of-sample Sharpe: {bench_sharpe}")
if hysys_sharpe > bench_sharpe:
    print('-> HYSYS-derived weighting outperforms the generic 3:2:1 spread out-of-sample.')
else:
    print('-> HYSYS-derived weighting does NOT outperform the generic 3:2:1 spread out-of-sample on Sharpe.')
    print('   (Report this honestly - it is still a valid and informative result.)')

In [ ]:
# ============================================================
# PLOTS - Equity curves, train/test boundary marked, both strategies
# ============================================================

fig = plt.figure(figsize=(14, 13))
gs  = gridspec.GridSpec(3, 1, hspace=0.35)

# 1. Equity curves - both strategies, full period
ax1 = fig.add_subplot(gs[0])
ax1.plot(df.index, df['equity_curve_hysys'], color='#2c3e50', linewidth=1.5, label='HYSYS partial spread strategy')
ax1.plot(df.index, df['equity_curve_321'],   color='#7f8c8d', linewidth=1.2, linestyle='--', label='3:2:1 benchmark strategy')
ax1.axvline(split_date, color='red', linestyle=':', linewidth=1.5, label='Train/Test split')
ax1.axhline(0, color='black', linewidth=0.8)
ax1.set_ylabel('Cumulative PnL ($/bbl)')
ax1.set_title('Equity Curves - HYSYS Strategy vs 3:2:1 Benchmark (train/test split marked)')
ax1.legend(fontsize=9)
ax1.grid(alpha=0.3)

# 2. Z-score with signals (HYSYS strategy)
ax2 = fig.add_subplot(gs[1])
ax2.plot(df.index, df['zscore'], color='#7f8c8d', linewidth=0.7, alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.axhline(ENTRY_THRESHOLD, color='#e74c3c', linewidth=1, linestyle='--', alpha=0.7)
ax2.axhline(-ENTRY_THRESHOLD, color='#27ae60', linewidth=1, linestyle='--', alpha=0.7)
ax2.axvline(split_date, color='red', linestyle=':', linewidth=1.5)
long_entries  = df[(df['signal'].diff() == 1)].index
short_entries = df[(df['signal'].diff() == -1)].index
ax2.scatter(long_entries,  df.loc[long_entries,  'zscore'], marker='^', color='#27ae60', s=35, zorder=5, label='Long entry')
ax2.scatter(short_entries, df.loc[short_entries, 'zscore'], marker='v', color='#e74c3c', s=35, zorder=5, label='Short entry')
ax2.set_ylabel('Z-score (HYSYS)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.3)

# 3. Drawdown comparison
dd_hysys = df['equity_curve_hysys'] - df['equity_curve_hysys'].cummax()
dd_321   = df['equity_curve_321']   - df['equity_curve_321'].cummax()
ax3 = fig.add_subplot(gs[2])
ax3.fill_between(df.index, dd_hysys, 0, alpha=0.4, color='#2c3e50', label='HYSYS strategy drawdown')
ax3.plot(df.index, dd_321, color='#7f8c8d', linewidth=1, linestyle='--', label='3:2:1 benchmark drawdown')
ax3.axvline(split_date, color='red', linestyle=':', linewidth=1.5)
ax3.set_ylabel('Drawdown ($/bbl)')
ax3.set_xlabel('Date')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.3)

plt.savefig('../data/backtest_results_full.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Save final results
results_table.to_csv('../data/performance_summary.csv', index=False)
df.to_csv('../data/backtest_results_full.csv')

print('Saved performance_summary.csv and backtest_results_full.csv')
print()
print('Copy the table below into the README results section:')
print()
print(results_table.to_markdown(index=False))